[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SubSurfObs/observatory_notebooks/blob/main/01_fdsn_access/index.ipynb)

# 01 — Multi-Network FDSN Data Access

This notebook demonstrates how to query station metadata and download seismic waveforms
from the University of Melbourne seismic network (VW) and the Seismology Research Centre
network (OZ) using FDSN web services.

**Worked example:** Mw ~3 earthquake near Moe, Victoria, 3 February 2026.

**Output:** waveform and inventory files saved to `data/` for use by subsequent notebooks.


<table style="border:1px solid #BFC3D1;border-radius:6px;padding:0.5rem 1rem;margin-bottom:1rem;font-size:0.9em;background:#f9f9fb;border-collapse:separate;">
<tr><td style="padding:2px 12px 2px 0"><strong>Authors</strong></td><td>Dan Sandiford&nbsp;<a href="https://orcid.org/0000-0002-2207-6837"><img src="https://orcid.org/sites/default/files/images/orcid_16x16.png" alt="ORCID" style="vertical-align:middle"> 0000-0002-2207-6837</a></td></tr>
<tr><td style="padding:2px 12px 2px 0"><strong>Institution</strong></td><td>University of Melbourne — Subsurface Observatory</td></tr>
<tr><td style="padding:2px 12px 2px 0"><strong>Funding</strong></td><td>AuScope / National Collaborative Research Infrastructure Strategy (NCRIS)</td></tr>
<tr><td style="padding:2px 12px 2px 0"><strong>Data</strong></td><td>VW network &middot; DOI <a href="https://doi.org/10.7914/8csc-8z27">10.7914/8csc-8z27</a></td></tr>
<tr><td style="padding:2px 12px 2px 0"><strong>Notebook DOI</strong></td><td><em>in preparation (Zenodo)</em></td></tr>
<tr><td style="padding:2px 12px 2px 0"><strong>Licence</strong></td><td><a href="https://creativecommons.org/licenses/by/4.0/">CC BY 4.0</a></td></tr>
</table>

In [ ]:
# ── Colab detection ──────────────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'obspy', 'cartopy'], check=True)
    # Download shared utilities
    subprocess.run(['wget', '-q',
        'https://raw.githubusercontent.com/SubSurfObs/observatory_notebooks/main/utils.py'],
        check=True)


In [ ]:
# ── Imports ──────────────────────────────────────────────────
from pathlib import Path
import sys

# Make utils.py importable whether running locally (from notebook subfolder)
# or from the repo root in CI
for p in [Path.cwd().parent, Path.cwd()]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from obspy import UTCDateTime
import matplotlib.pyplot as plt

from utils import (
    Provider, collect_stations_and_waveforms,
    FDSN_UOM, FDSN_AUSPASS, FDSN_RS, FDSN_IRIS,
)
print('Imports OK')


In [ ]:
# ── Event parameters ─────────────────────────────────────────
# Mw ~3, Moe, Victoria — 3 Feb 2026
EVENT_LAT  =  -38.18
EVENT_LON  =  146.35
EVENT_TIME =  UTCDateTime('2026-02-03T08:12:00')

# Time window around the event
T_PRE  = 30   # seconds before origin time
T_POST = 90   # seconds after origin time
T_START = EVENT_TIME - T_PRE
T_END   = EVENT_TIME + T_POST

# Networks and search radius
NETWORKS   = ['VW', 'OZ']
MAX_RADIUS = 3.0   # degrees

print(f'Window: {T_START} → {T_END}')


## Station inventory

Query both providers for stations within `MAX_RADIUS` degrees of the event.
The resolver tries providers in priority order (UoM first, AusPass second) and
assigns each network to the first provider that serves it.


In [ ]:
providers = [
    Provider('UoM',     FDSN_UOM),      # VW, VX, Z1
    Provider('AusPass', FDSN_AUSPASS),  # OZ, AU, and national archive
    Provider('IRIS',    FDSN_IRIS),     # global fallback
]

st, selections = collect_stations_and_waveforms(
    providers=providers,
    requested_networks=NETWORKS,
    latitude=EVENT_LAT,
    longitude=EVENT_LON,
    maxradius=MAX_RADIUS,
    starttime=T_START,
    endtime=T_END,
    channel='*H*,*N*',     # broadband + strong-motion
    attach_response=True,
)

# Collect the merged inventory
from obspy.core.inventory import Inventory
inv_all = Inventory()
for sel in selections.values():
    inv_all += sel.inventory

print(f'Stations: {len(list(inv_all.get_contents()["stations"]))}  |  Traces: {len(st)}')
for name, sel in selections.items():
    print(f'  {name}: networks={sel.networks}, stations={sel.stations}')


In [ ]:
# ── Station map (cartopy) ─────────────────────────────────────
import cartopy.crs as ccrs
import cartopy.feature as cfeature

lons, lats, labels = [], [], []
for net in inv_all:
    for sta in net:
        if sta.longitude is not None and sta.latitude is not None:
            lons.append(sta.longitude)
            lats.append(sta.latitude)
            labels.append(sta.code)

pad = 0.5
extent = [min(lons) - pad, max(lons) + pad, min(lats) - pad, max(lats) + pad]

fig, ax = plt.subplots(figsize=(8, 6),
                        subplot_kw={'projection': ccrs.PlateCarree()})
ax.set_extent(extent, crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND,   facecolor='#f5f5f0')
ax.add_feature(cfeature.OCEAN,  facecolor='#dce9f5')
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS,   linewidth=0.4, linestyle=':')
ax.gridlines(draw_labels=True, linewidth=0.4, color='grey', alpha=0.5)

ax.scatter(lons, lats, transform=ccrs.PlateCarree(),
           marker='^', s=80, color='#000F46', zorder=5, label='Station')
for lon, lat, lab in zip(lons, lats, labels):
    ax.text(lon + 0.05, lat, lab, transform=ccrs.PlateCarree(), fontsize=7)

ax.scatter([EVENT_LON], [EVENT_LAT], transform=ccrs.PlateCarree(),
           marker='*', s=200, color='#E5007D', zorder=6, label='Event')

ax.set_title('Station map — VW + OZ networks', fontsize=11)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


## Waveforms

The merged stream contains all channels returned by both providers.
Below we select the vertical broadband channel (`*HZ`) for display.


In [ ]:
st_Z = st.select(channel='*HZ').copy()
st_Z.detrend('demean')
st_Z.filter('bandpass', freqmin=1.0, freqmax=20.0, corners=4)
print(st_Z)


In [ ]:
fig = st_Z.plot(equal_scale=False, size=(900, 600), handle=True)
fig.suptitle('VW + OZ — *HZ — bandpass 1–20 Hz', fontsize=11)
plt.tight_layout()
plt.show()


## Save data

Write the full (unfiltered) stream and inventory to `data/` for use by
notebook 02 (phase picking). We save all channels so that downstream
notebooks can choose their own pre-processing.


In [ ]:
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)

mseed_path = data_dir / 'ex01.mseed'
xml_path   = data_dir / 'ex01.xml'

st.write(str(mseed_path), format='MSEED')
inv_all.write(str(xml_path), format='STATIONXML')

print(f'Saved {mseed_path}  ({mseed_path.stat().st_size / 1024:.0f} kB)')
print(f'Saved {xml_path}    ({xml_path.stat().st_size / 1024:.0f} kB)')


---

**Next:** [02 — Phase Picking with SeisBench](../02_phase_picking/index.ipynb)

**Data source:** University of Melbourne Seismic Network (VW),
DOI [10.7914/8csc-8z27](https://doi.org/10.7914/8csc-8z27).  
**Licence:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
